In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!pip install transformers
!pip install datasets
!pip install torchaudio
!pip install evaluate
!pip install numpy

In [10]:
# ==========================================
# STAGE 0: Environment & Dependency Setup
# ==========================================
import os
import torch
import evaluate
import numpy as np
import warnings
from datasets import Dataset, DatasetDict, Audio
from transformers import (
    AutoFeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

warnings.filterwarnings("ignore")

try:
    PROJECT_DIR = '/content/drive/MyDrive/DeepFake_Audio_Detection'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print(f"Checkpoints will be safely saved to: {PROJECT_DIR}")
except ImportError:
    PROJECT_DIR = './DeepFake_Audio_Detection'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print(f"Running locally. Checkpoints will be saved to: {PROJECT_DIR}")

# ==========================================
# STAGE 1: Robust Dataset Loading (Folder-Specific)
# ==========================================
MODEL_ID = "facebook/wav2vec2-base"
SAMPLE_RATE = 16000
MAX_DURATION = 2.0

DATA_DIR = '/content/drive/MyDrive/Datasets/for-2seconds'

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"CRITICAL ERROR: Cannot find the path {DATA_DIR}. Please check if Google Drive is mounted and the spelling/capitalization is exact.")
else:
    print(f"Path confirmed. Searching inside: {DATA_DIR}")

def load_split(split_dir):
    file_paths = []
    labels = []
    label_map = {"real": 0, "fake": 1}

    for label_str, label_idx in label_map.items():
        folder_path = os.path.join(split_dir, label_str)
        if not os.path.exists(folder_path):
            print(f"  -> Warning: Subdirectory not found - {folder_path}")
            continue

        for root, _, files in os.walk(folder_path):
            for file in files:
                if file.lower().endswith(('.wav', '.mp3', '.flac', '.m4a', '.ogg')):
                    full_path = os.path.join(root, file)
                    file_paths.append(full_path)
                    labels.append(label_idx)

    if not file_paths:
        return None

    print(f"  -> Successfully loaded {len(file_paths)} audio files from {os.path.basename(split_dir)}.")
    dataset = Dataset.from_dict({"audio": file_paths, "label": labels})
    dataset = dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    return dataset

split_names = ['training', 'validation', 'testing']
datasets_dict = {}

for split in split_names:
    split_path = os.path.join(DATA_DIR, split)
    print(f"\nScanning '{split}' directory...")

    if os.path.exists(split_path):
        ds = load_split(split_path)
        if ds:
            hf_split_name = 'train' if split == 'training' else ('test' if split == 'testing' else split)
            datasets_dict[hf_split_name] = ds
    else:
        print(f"  -> Skipping. Split directory does not exist: {split_path}")

full_dataset = DatasetDict(datasets_dict)
print("\nDataset Loading Complete! Structure:")
print(full_dataset)

if len(full_dataset.keys()) == 0:
    raise ValueError("CRITICAL ERROR: Dataset is completely empty. The script has halted. Please verify your Google Drive is properly mounted and the folders contain audio files.")


Checkpoints will be safely saved to: /content/drive/MyDrive/DeepFake_Audio_Detection
Path confirmed. Searching inside: /content/drive/MyDrive/Datasets/for-2seconds

Scanning 'training' directory...
  -> Successfully loaded 13997 audio files from training.

Scanning 'validation' directory...
  -> Successfully loaded 2826 audio files from validation.

Scanning 'testing' directory...
  -> Successfully loaded 1088 audio files from testing.

Dataset Loading Complete! Structure:
DatasetDict({
    train: Dataset({
        features: ['audio', 'label'],
        num_rows: 13997
    })
    validation: Dataset({
        features: ['audio', 'label'],
        num_rows: 2826
    })
    test: Dataset({
        features: ['audio', 'label'],
        num_rows: 1088
    })
})


In [ ]:
# ==========================================
# STAGE 2: Accelerated Feature Extraction
# ==========================================
import os
import torch
import numpy as np
import random
from transformers import AutoFeatureExtractor

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

num_cores = os.cpu_count() or 1

def apply_audio_dropout(audio_array, drop_prob=0.1, max_drop_size=1000):
    if random.random() > drop_prob:
        return audio_array

    audio_len = len(audio_array)
    drop_size = random.randint(100, max_drop_size)
    start_idx = random.randint(0, audio_len - drop_size)

    augmented_array = np.copy(audio_array)
    augmented_array[start_idx : start_idx + drop_size] = 0.0

    return augmented_array

def preprocess_function(examples, is_training=False):
    audio_arrays = [x["array"] for x in examples["audio"]]

    if is_training:
        audio_arrays = [apply_audio_dropout(a) for a in audio_arrays]

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLE_RATE,
        max_length=int(SAMPLE_RATE * MAX_DURATION),
        truncation=True,
        padding="max_length"
    )
    return inputs

print(f"Processing Training Data (Utilizing {num_cores} CPU Cores)...")
encoded_train = full_dataset["train"].map(
    lambda x: preprocess_function(x, is_training=True),
    remove_columns=["audio"],
    batched=True,
    batch_size=100,
    num_proc=num_cores
)

print(f"Processing Validation Data (Utilizing {num_cores} CPU Cores)...")
encoded_validation = full_dataset["validation"].map(
    lambda x: preprocess_function(x, is_training=False),
    remove_columns=["audio"],
    batched=True,
    batch_size=100,
    num_proc=num_cores
)

print(f"Processing Test Data (Utilizing {num_cores} CPU Cores)...")
encoded_test = full_dataset["test"].map(
    lambda x: preprocess_function(x, is_training=False),
    remove_columns=["audio"],
    batched=True,
    batch_size=100,
    num_proc=num_cores
)


# ==========================================
# STAGE 3: Model & Metrics Initialization
# ==========================================
id2label = {0: "REAL", 1: "FAKE"}
label2id = {"REAL": 0, "FAKE": 1}

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    label2id=label2id,
    id2label=id2label
)

model.freeze_feature_encoder()

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=eval_pred.label_ids)
    f1 = f1_metric.compute(predictions=predictions, references=eval_pred.label_ids)
    return {**accuracy, **f1}

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Processing Training Data (Utilizing 2 CPU Cores)...


Map (num_proc=2):   0%|          | 0/13997 [00:00<?, ? examples/s]

Processing Validation Data (Utilizing 2 CPU Cores)...


Map (num_proc=2):   0%|          | 0/2826 [00:00<?, ? examples/s]

Processing Test Data (Utilizing 2 CPU Cores)...


Map (num_proc=2):   0%|          | 0/1088 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# STAGE 4: Anti-Overfitting Training
# ==========================================
from torch import nn
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_DIR, "wav2vec2-generalized"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,

    weight_decay=0.05,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    warmup_steps=500,
    fp16=True,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_validation,
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

print("Starting Generalized Training...")
trainer.train()

print("Evaluating Final Model...")
final_metrics = trainer.evaluate(encoded_test)
print(f"Final Test Metrics: {final_metrics}")

robust_model_path = os.path.join(PROJECT_DIR, "best_generalized_model")
trainer.save_model(robust_model_path)
print(f"Generalized model saved to: {robust_model_path}")

In [ ]:
# ==========================================
# STAGE 5: Real-World Inference Engine
# ==========================================
import torch
import torchaudio
import torch.nn.functional as F
from transformers import AutoFeatureExtractor, Wav2Vec2ForSequenceClassification

MODEL_PATH = "/content/drive/MyDrive/DeepFake_Audio_Detection/best_generalized_model"

print("Loading model and processor into memory...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_PATH)
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict_audio(file_path):
    print(f"\nAnalyzing: {file_path}")

    try:
        waveform, sample_rate = torchaudio.load(file_path)
    except Exception as e:
        return f"Error loading audio: {e}"

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)

    audio_array = waveform.squeeze().numpy()

    inputs = feature_extractor(
        audio_array,
        sampling_rate=16000,
        max_length=32000,
        truncation=True,
        padding="max_length",
        return_tensors="pt" # Return PyTorch tensors
    )

    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

        probabilities = F.softmax(logits, dim=-1)

        predicted_class_id = torch.argmax(probabilities, dim=-1).item()
        confidence = probabilities[0][predicted_class_id].item()

    label = model.config.id2label[predicted_class_id]

    return label, confidence

# Test Your Audio Here

TEST_AUDIO_PATH = "/content/Recording (6).m4a"

label, confidence = predict_audio(TEST_AUDIO_PATH)

print("-" * 30)
print(f"Prediction:  {label}")
print(f"Confidence:  {confidence * 100:.2f}%")
print("-" * 30)

In [ ]:
import os
from google.colab import files
import shutil


model_save_path = os.path.join(PROJECT_DIR, "best_deepfake_model")

zip_filename_desired = "best_deepfake_model.zip"

output_zip_base_path = os.path.join("/content/", os.path.splitext(zip_filename_desired)[0])

print(f"Creating zip archive of '{model_save_path}' to '{output_zip_base_path}.zip'")


archive_full_path = shutil.make_archive(
    output_zip_base_path,
    'zip',
    root_dir=PROJECT_DIR,
    base_dir=os.path.basename(model_save_path)
)

print(f"Your model '{archive_full_path}' is ready for download.")
files.download(archive_full_path)


In [ ]:
import os
import torch
import torchaudio
from transformers import AutoFeatureExtractor, Wav2Vec2ForSequenceClassification

audio_file_path = '/content/Recording (6).m4a'
pytorch_model_path = robust_model_path

MODEL_ID = "facebook/wav2vec2-base"
SAMPLE_RATE = 16000
MAX_DURATION = 2.0
id2label = {0: "REAL", 1: "FAKE"}

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

print(f"Audio file to test: {audio_file_path}")
print(f"PyTorch model to load from: {pytorch_model_path}")

In [ ]:
print("Loading PyTorch model...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(pytorch_model_path)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"PyTorch model loaded and moved to {device}.")

In [ ]:
print(f"Loading and preprocessing audio file: {audio_file_path}")

if not os.path.exists(audio_file_path):
    raise FileNotFoundError(f"Error: Audio file not found at {audio_file_path}")

sound, sr = torchaudio.load(audio_file_path)

if sr != SAMPLE_RATE:
    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
    sound = resampler(sound)

if sound.shape[0] > 1:
    sound = sound[0].unsqueeze(0)


inputs = feature_extractor(
    sound.squeeze().numpy(),
    sampling_rate=SAMPLE_RATE,
    max_length=int(SAMPLE_RATE * MAX_DURATION),
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

inputs = {k: v.to(device) for k, v in inputs.items()}

print("Audio preprocessing complete.")

In [ ]:
# Make a prediction
print("Making prediction...")
with torch.no_grad():
    logits = model(**inputs).logits

# Apply softmax to get probabilities
probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
predicted_class_id = torch.argmax(logits, dim=1).item()
predicted_class_label = id2label[predicted_class_id]

# Display results
print("\n--- Prediction Results ---")
print(f"Audio file: {audio_file_path}")
print(f"Predicted Label: {predicted_class_label}")
print(f"Confidence (REAL): {probabilities[0]:.4f}")
print(f"Confidence (FAKE): {probabilities[1]:.4f}")
print("--------------------------")
